## 1) Objective

This notebook profiles Instacart raw datasets before data preparation and EDA.

Questions answered:
- How big is each table?
- What is the schema and data type quality?
- Where are missing values and duplicates?
- Are keys and relationships consistent across tables?

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display

DATA_PATH = '../../data/raw/'

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.4f}'.format)

In [2]:
# Load all raw datasets
order_products_prior = pd.read_csv(f'{DATA_PATH}order_products__prior.csv')
order_products_train = pd.read_csv(f'{DATA_PATH}order_products__train.csv')
orders = pd.read_csv(f'{DATA_PATH}orders.csv')
products = pd.read_csv(f'{DATA_PATH}products.csv')
aisles = pd.read_csv(f'{DATA_PATH}aisles.csv')
departments = pd.read_csv(f'{DATA_PATH}departments.csv')

tables = {
    'orders': orders,
    'order_products_prior': order_products_prior,
    'order_products_train': order_products_train,
    'products': products,
    'aisles': aisles,
    'departments': departments
}

## 2) Table Size Overview

In [3]:
overview = pd.DataFrame({
    'table': list(tables.keys()),
    'rows': [df.shape[0] for df in tables.values()],
    'cols': [df.shape[1] for df in tables.values()],
    'memory_mb': [round(df.memory_usage(deep=True).sum() / 1024**2, 2) for df in tables.values()]
}).sort_values('rows', ascending=False)

overview

,table,rows,cols,memory_mb
1,order_products_prior,32434489,4,989.8200
0,orders,3421083,7,332.7100
2,order_products_train,1384617,4,42.2600
3,products,49688,4,4.9300
4,aisles,134,2,0.0100
5,departments,21,2,0.0000


## 3) Schema Check
For each table, inspect sample rows and data types.

In [4]:
for name, df in tables.items():
    print(f'\n=== {name} ===')
    display(df.head(3))
    display(df.dtypes.rename('dtype').to_frame())


=== orders ===


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0000
2,473747,1,prior,3,3,12,21.0000


,dtype
order_id,int64
user_id,int64
eval_set,str
order_number,int64
order_dow,int64
order_hour_of_day,int64
days_since_prior_order,float64



=== order_products_prior ===


,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0


...

,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars


,dtype
aisle_id,int64
aisle,str



=== departments ===


,department_id,department
0,1,frozen
1,2,other
2,3,bakery


,dtype
department_id,int64
department,str


## 4) Data Quality Summary
Check missing values and duplicated records.

In [5]:
quality_rows = []
for name, df in tables.items():
    missing_pct = (df.isna().mean() * 100).round(2)
    quality_rows.append({
        'table': name,
        'duplicate_rows': int(df.duplicated().sum()),
        'total_missing_cells': int(df.isna().sum().sum()),
        'max_missing_col_pct': float(missing_pct.max())
    })

quality_summary = pd.DataFrame(quality_rows).sort_values('total_missing_cells', ascending=False)
quality_summary

,table,duplicate_rows,total_missing_cells,max_missing_col_pct
0,orders,0,206209,6.0300
1,order_products_prior,0,0,0.0000
2,order_products_train,0,0,0.0000
3,products,0,0,0.0000
4,aisles,0,0,0.0000
5,departments,0,0,0.0000


## 5) Key Integrity Checks
Validate uniqueness and cross-table key consistency.

In [6]:
checks = {}
checks['orders.order_id_unique'] = orders['order_id'].is_unique
checks['products.product_id_unique'] = products['product_id'].is_unique
checks['aisles.aisle_id_unique'] = aisles['aisle_id'].is_unique
checks['departments.department_id_unique'] = departments['department_id'].is_unique

checks['prior.order_id_in_orders_pct'] = round(order_products_prior['order_id'].isin(orders['order_id']).mean() * 100, 2)
checks['train.order_id_in_orders_pct'] = round(order_products_train['order_id'].isin(orders['order_id']).mean() * 100, 2)
checks['prior.product_id_in_products_pct'] = round(order_products_prior['product_id'].isin(products['product_id']).mean() * 100, 2)
checks['train.product_id_in_products_pct'] = round(order_products_train['product_id'].isin(products['product_id']).mean() * 100, 2)

pd.Series(checks, name='result')

orders.order_id_unique                 True
products.product_id_unique             True
aisles.aisle_id_unique                 True
departments.department_id_unique       True
prior.order_id_in_orders_pct       100.0000
train.order_id_in_orders_pct       100.0000
prior.product_id_in_products_pct   100.0000
train.product_id_in_products_pct   100.0000
Name: result, dtype: object

## 6) Quick Business Sanity Checks

In [7]:
display(orders['eval_set'].value_counts(dropna=False).to_frame('count'))
display(orders['order_dow'].value_counts().sort_index().to_frame('count'))
display(orders['order_hour_of_day'].value_counts().sort_index().to_frame('count').head())
display(order_products_prior['reordered'].value_counts(dropna=False).to_frame('count'))

,count
eval_set,
prior,3214874
train,131209
test,75000


,count
order_dow,
0,600905
1,587478
2,467260
3,436972
4,426339
5,453368
6,448761


,count
order_hour_of_day,
0,22758
1,12398
2,7539
3,5474
4,5527


,count
reordered,
1,19126536
0,13307953


## 7) Findings and Next Actions
After running all cells, write 4-6 bullets:
- Biggest table and what it implies for processing
- Most important missing-value issue
- Integrity check result summary
- One behavioral insight from order timing/reordered
- Risks before feature engineering

In [8]:
order_products_prior.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 32434489 entries, 0 to 32434488
Data columns (total 4 columns):
 #   Column             Dtype
---  ------             -----
 0   order_id           int64
 1   product_id         int64
 2   add_to_cart_order  int64
 3   reordered          int64
dtypes: int64(4)
memory usage: 989.8 MB
